# Predicting Income with Social Data
## Solution Notebook — Linear Regression in R (PSID 2017)

**Goal:** Complete reference implementation of the linear-regression workflow on PSID data: assumptions checking, cleaning, train/test split, simple & multiple models, fit assessment, coefficient interpretation, alternates, more practice, and simulation.

**Flowchart:**

![Predicting Income Pipeline](predicting_income_flowchart.png)

---
### How to use
Run cells top-to-bottom. Compare with the Practice Skeleton when you want to try the exercises yourself first.

A detailed **Cheat Sheet** is included at the bottom.

## 0. Setup

In [ ]:
library(dplyr)
library(ggplot2)

psid <- read.csv("data/psid_2017.csv")
head(psid)
cat("Raw n:", nrow(psid), "\n")

## 1. Clean and check data assumptions

In [ ]:
# Task 1
str(psid)

In [ ]:
# Task 2 — age raw
age_raw_plot <- psid %>%
  ggplot(aes(age)) +
  geom_histogram(binwidth = 5, fill = "#1f77b4", color = "white") +
  labs(title = "Age Distribution (Raw)", x = "Age", y = "Count") +
  theme_minimal()
age_raw_plot
# Note: max age = 999 is clearly a missing/code value.

In [ ]:
# Task 3 — filter working age
psid_age <- psid %>% filter(age >= 18, age <= 75)
cat("After age filter:", nrow(psid_age), "\n")

In [ ]:
# Task 4
age_clean_plot <- psid_age %>%
  ggplot(aes(age)) +
  geom_histogram(binwidth = 5, fill = "#2ca02c", color = "white") +
  labs(title = "Age Distribution (18-75)", x = "Age", y = "Count") +
  theme_minimal()
age_clean_plot

In [ ]:
# Task 5 — education boxplot
educ_box <- psid_age %>%
  ggplot(aes(x = "", y = education_years)) +
  geom_boxplot(fill = "#ff7f0e") +
  labs(title = "Education Years (before filter)", x = "", y = "Years") +
  theme_minimal()
educ_box
# 99 appears as an extreme outlier → coded missing

In [ ]:
# Task 6
psid_clean <- psid_age %>%
  filter(education_years >= 5, education_years <= 25)
cat("After education filter:", nrow(psid_clean), "\n")

In [ ]:
# Task 7
income_box <- psid_clean %>%
  ggplot(aes(x = "", y = labor_income)) +
  geom_boxplot(fill = "#d62728") +
  labs(title = "Labor Income", x = "", y = "Yearly labor income ($)") +
  theme_minimal() +
  scale_y_continuous(labels = scales::comma)
income_box

In [ ]:
# Task 8
summary(psid_clean$labor_income)
cat("Share with positive income:", round(mean(psid_clean$labor_income > 0), 3), "\n")
# Heavy mass at zero → many non-earners / students / homemakers / disabled.

In [ ]:
# Task 9 — mean income by age + final filter to positive earners
mean_by_age <- psid_clean %>%
  group_by(age) %>%
  summarise(mean_income = mean(labor_income, na.rm = TRUE), .groups = "drop")

ggplot(mean_by_age, aes(age, mean_income)) +
  geom_point(color = "#1f77b4") +
  labs(title = "Mean Labor Income by Age (incl. zeros)", x = "Age", y = "Mean income") +
  theme_minimal() +
  scale_y_continuous(labels = scales::comma)

# Restrict to positive labor income for clearer linear-regression teaching results
psid_clean <- psid_clean %>% filter(labor_income > 0)
cat("Final analytic n (age 18-75, edu 5-25, income > 0):", nrow(psid_clean), "\n")

# Convert gender to factor with labels
psid_clean <- psid_clean %>%
  mutate(gender = factor(gender, levels = c(1, 2), labels = c("Male", "Female")))

## 2. Build model and assess fit

In [ ]:
# Task 10 — train/test
set.seed(123)
sample_idx <- sample(c(TRUE, FALSE), nrow(psid_clean), replace = TRUE, prob = c(0.6, 0.4))
train <- psid_clean[sample_idx, ]
test  <- psid_clean[!sample_idx, ]
cat("Train n:", nrow(train), "  Test n:", nrow(test), "\n")

In [ ]:
# Task 11 — simple LM
model <- lm(labor_income ~ education_years, data = train)
summary(model)
# Positive slope as expected; R² modest because income has large residual variance.

In [ ]:
# Task 12 — scatter + lm + loess
ggplot(train, aes(education_years, labor_income)) +
  geom_point(alpha = 0.3, color = "#1f77b4") +
  geom_smooth(method = "lm", se = TRUE, color = "red") +
  geom_smooth(method = "loess", se = FALSE, color = "darkgreen", linetype = "dashed") +
  labs(title = "Labor Income ~ Education Years (Train)",
       subtitle = "Red = OLS line, Green dashed = LOESS",
       x = "Education years", y = "Labor income ($)") +
  theme_minimal() +
  scale_y_continuous(labels = scales::comma)

In [ ]:
# Task 13
r_sq <- summary(model)$r.squared * 100
r_sq

In [ ]:
# Task 14
sprintf("Based on a simple linear regression, approximately %.1f percent of the variation in labor income can be explained by years of formal education alone.", r_sq)

## 3. Comparison model and results

In [ ]:
# Task 15 — multiple LM
model_2 <- lm(labor_income ~ education_years + age + gender, data = train)
summary(model_2)

In [ ]:
# Task 16
r_sq_2 <- summary(model_2)$r.squared * 100
r_sq_2

In [ ]:
# Task 17
sprintf("Adding age and gender raises the explained variation to approximately %.1f percent.", r_sq_2)

In [ ]:
# Task 18 — observed vs predicted
test$pred <- predict(model_2, newdata = test)

ggplot(test, aes(age, labor_income)) +
  geom_point(alpha = 0.25, color = "gray40") +
  geom_line(aes(y = pred), color = "blue", linewidth = 0.9) +
  labs(title = "Observed vs Predicted (Model 2 on Test Set)",
       x = "Age", y = "Labor income ($)") +
  theme_minimal() +
  scale_y_continuous(labels = scales::comma)

In [ ]:
# Task 19 — coefficient table
print(summary(model_2)$coefficients)
cat("\nInterpretation notes:\n")
cat("- education_years, age and genderFemale are all statistically significant (p << 0.05).\n")
cat("- genderFemale coefficient ≈ difference in expected income for females vs males (reference), holding education and age fixed.\n")
cat("- age has a large positive association (each additional year of age ≈ +$900 in this sample of earners).\n")

In [ ]:
# Task 20
education_coefficient <- coef(model_2)["education_years"]
education_coefficient

In [ ]:
# Task 21
sprintf("Holding age and gender constant, each additional year of formal education is associated with an estimated $%.0f increase in annual labor income.", education_coefficient)

### Task 22 — Summary of findings
- After cleaning (age 18-75, education 5-25 years, positive labor income) we retain ~1 600 observations.
- Simple model R² ≈ 3 %; multiple model R² ≈ 13 %.
- Education remains positively and significantly associated with income after controlling for age and gender.
- Female earners have lower predicted income than male earners of the same age and education (gender gap).
- Residual variance is still large → many other factors (occupation, hours, industry, region, experience) are not in the model.

---
## Alternate Code Paths

In [ ]:
# Alternate 1: pure base-R filtering
psid_alt <- psid[psid$age >= 18 & psid$age <= 75 &
                 psid$education_years >= 5 & psid$education_years <= 25 &
                 psid$labor_income > 0, ]
psid_alt$gender <- factor(psid_alt$gender, levels = c(1,2), labels = c("Male","Female"))
cat("Base-R filter n:", nrow(psid_alt), "\n")

# Alternate 2: adjusted R² and RSE percentage
adj_r2 <- summary(model_2)$adj.r.squared
rse_pct <- sigma(model_2) / mean(train$labor_income) * 100
cat("Adjusted R²:", round(adj_r2, 4), "\n")
cat("RSE as % of mean income:", round(rse_pct, 1), "%\n")

# Alternate 3: add a college dummy and re-fit
train$college <- ifelse(train$education_years >= 16, 1, 0)
model_college <- lm(labor_income ~ college + age + gender, data = train)
cat("College dummy coefficient:", coef(model_college)["college"], "\n")

---
## More Practice (worked examples)

In [ ]:
# 1. Add married (binary)
train$married_bin <- factor(ifelse(train$married == 1, "Married", "NotMarried"))
model_3 <- lm(labor_income ~ education_years + age + gender + married_bin, data = train)
cat("R² with married:", round(summary(model_3)$r.squared, 4), "\n")
print(coef(model_3))

# 2. Residual diagnostics
par(mfrow = c(1, 2))
hist(residuals(model_2), main = "Residuals of Model 2", col = "lightblue", breaks = 40)
plot(fitted(model_2), residuals(model_2), main = "Residuals vs Fitted",
     xlab = "Fitted", ylab = "Residual", pch = 16, cex = 0.4, col = rgb(0,0,0,0.3))
abline(h = 0, col = "red")
par(mfrow = c(1, 1))

# 3. Test-set MSE
test$pred2 <- predict(model_2, newdata = test)
mse <- mean((test$labor_income - test$pred2)^2)
cat("Test MSE:", round(mse, 0), "\n")

---
## Simulation — Sensitivity

In [ ]:
n_sim         <- 40
sample_frac   <- 0.6
noise_sd_mult <- 1.0
min_edu       <- 5
max_edu       <- 25
age_lo        <- 18
age_hi        <- 75

set.seed(42)
sim_mat <- replicate(n_sim, {
  d <- psid %>%
    filter(age >= age_lo, age <= age_hi,
           education_years >= min_edu, education_years <= max_edu,
           labor_income > 0) %>%
    mutate(gender = factor(gender, levels = c(1,2), labels = c("Male","Female")))
  idx <- sample(c(TRUE, FALSE), nrow(d), replace = TRUE, prob = c(sample_frac, 1 - sample_frac))
  tr <- d[idx, ]
  if (noise_sd_mult != 1) {
    tr$labor_income <- tr$labor_income +
      rnorm(nrow(tr), 0, sd(tr$labor_income) * (noise_sd_mult - 1))
  }
  m <- lm(labor_income ~ education_years + age + gender, data = tr)
  c(r2 = summary(m)$r.squared, edu_coef = unname(coef(m)["education_years"]))
})

sim_df <- as.data.frame(t(sim_mat))
cat("Mean R²:", round(mean(sim_df$r2), 4), "\n")
cat("SD of R²:", round(sd(sim_df$r2), 4), "\n")
cat("Mean education coefficient: $", round(mean(sim_df$edu_coef), 0), "\n", sep = "")
cat("95% interval for education coef: $",
    round(quantile(sim_df$edu_coef, 0.025), 0), " – $",
    round(quantile(sim_df$edu_coef, 0.975), 0), "\n", sep = "")

par(mfrow = c(1, 2))
hist(sim_df$r2, main = "Simulated R²", xlab = "R²", col = "skyblue", breaks = 12)
hist(sim_df$edu_coef, main = "Simulated Education Coefficient",
     xlab = "Coefficient ($)", col = "salmon", breaks = 12)
par(mfrow = c(1, 1))

---
## Cheat Sheet — Linear Regression in R (PSID / Social Data)

### Data inspection & cleaning
```r
str(df)                          # structure
summary(df$var)                  # five-number + mean
table(df$cat, useNA = "ifany")   # frequency incl. NA
df %>% filter(cond1, cond2)      # dplyr filter
df[df$age >= 18 & df$age <= 75, ]# base-R equivalent
```

### Visualization (ggplot2)
```r
ggplot(df, aes(x)) + geom_histogram(binwidth = 5)
ggplot(df, aes(x = "", y = var)) + geom_boxplot()
ggplot(df, aes(x, y)) + geom_point(alpha = 0.3) +
  geom_smooth(method = "lm") + geom_smooth(method = "loess", se = FALSE)
```

### Train / test split
```r
set.seed(123)
idx <- sample(c(TRUE, FALSE), nrow(df), replace = TRUE, prob = c(0.6, 0.4))
train <- df[idx, ];  test <- df[!idx, ]
```

### Modeling
```r
model  <- lm(y ~ x, data = train)                 # simple
model2 <- lm(y ~ x1 + x2 + factor(x3), data = train)  # multiple
summary(model)                                   # coefficients, R², RSE, p-values
summary(model)$r.squared
coef(model)["x1"]
sigma(model)                                     # residual SE
predict(model, newdata = test)                   # predictions
```

### Fit metrics & interpretation
- **R²** = proportion of variance in y explained by the model (0–1).
- **RSE / mean(y)** ≈ average percentage error.
- Continuous coefficient: “holding other variables constant, a 1-unit increase in X is associated with a β change in Y.”
- Binary factor coefficient: “the difference in expected Y between the level and the reference level.”

### Common pitfalls with survey data
- Many zeros → consider filtering to positive earners or using a two-part / log model.
- Coded missing (99, 999, 9) → always inspect ranges and `table()`.
- Factor vs numeric: convert categorical predictors with `factor()` and meaningful labels.

### Audience adaptation (quick checklist)
| Audience        | Depth              | Language                     | Visuals              |
|-----------------|--------------------|------------------------------|----------------------|
| Expert / Tech   | Full coefficients, SE, p, residual diagnostics | Statistical terms OK | Residual plots, R² comparison |
| Executive       | Headline R², key β, policy implication | Minimal jargon            | One clear scatter + line |
| Nonspecialist   | “Each extra year of school ≈ $X more income” | Everyday words            | Simple bar or annotated scatter |

---
*End of Solution Notebook.*